<a href="https://colab.research.google.com/github/opherdonchin/BayesShortCourse/blob/main/sleep/solved/06_lognormal_regression.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Sleep deprivation 6 — Lognormal regression

Changing the likelihood changes the scale on which parameters and priors live. We first demonstrate why blindly reusing Gaussian priors fails, then fit a lognormal regression with priors expressed on the log-reaction-time scale.

## Setup

This course pins PyMC, modular ArviZ, and Bambi for reproducibility because their APIs can change across major versions.

In [ ]:
%pip install -q \
    "pandas==2.2.3" \
    "pymc==6.3.2" \
    "arviz-base==1.3.0" \
    "arviz-stats==1.3.2" \
    "arviz-plots[matplotlib]==1.3.1" \
    "bambi==0.21.0"

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import bambi as bmb
import pymc as pm
import arviz_base as azb
import arviz_stats as azs
import arviz_plots as azp

RANDOM_SEED = 20260924
azp.style.use("arviz-variat")

print("PyMC:", pm.__version__)
print("Bambi:", bmb.__version__)
print("arviz-base:", azb.__version__)
print("arviz-plots:", azp.__version__)
print("arviz-stats:", azs.__version__)

## Data

The original study contains two adaptation/training days followed by a baseline measurement and then seven nights of severe sleep restriction. Following the chapter, we drop original days 0–1 and subtract 2 from the remaining day number. Therefore **`Days = 0` is the baseline measurement before sleep deprivation begins**.

That zero point is scientifically meaningful, so every Bambi model in this sequence uses `center_predictors=False`. The `Intercept` prior is therefore a prior on baseline reaction time rather than reaction time at the average deprivation day.

In [ ]:
DATA_URL = "https://raw.githubusercontent.com/vincentarelbundock/Rdatasets/master/csv/lme4/sleepstudy.csv"

sleep = pd.read_csv(DATA_URL).drop(columns="rownames")
sleep = sleep.loc[sleep["Days"] >= 2].copy()
sleep["Days"] = sleep["Days"] - 2
sleep["Subject"] = sleep["Subject"].astype(str)
sleep = sleep.reset_index(drop=True)

print(f"{sleep['Subject'].nunique()} participants, {len(sleep)} observations")
print(f"Days: {sleep['Days'].min()} to {sleep['Days'].max()}")
sleep.head()

# 6.1 Priors on the wrong scale

What goes wrong if Gaussian-scale priors are reused in a lognormal model?

For a lognormal likelihood, `mu` is the location parameter of **log reaction time**. An intercept around 250 is therefore astronomically large. This is exactly the sort of mistake prior predictive simulation should catch.

In [ ]:
bad_priors = {
    "Intercept": bmb.Prior("Normal", mu=250, sigma=100),
    "Days": bmb.Prior("Normal", mu=0, sigma=20),
    "sigma": bmb.Prior("Exponential", lam=0.02),
}
bad_model = bmb.Model(
    "Reaction ~ Days", sleep, family="lognormal", priors=bad_priors, center_predictors=False
)
bad_prior = bad_model.prior_predictive(draws=200, random_seed=RANDOM_SEED)

yrep = bad_prior["prior_predictive"]["Reaction"].to_numpy()
finite_fraction = np.isfinite(yrep).mean()
finite_values = yrep[np.isfinite(yrep) & (yrep > 0)]
print(f"Finite prior-predictive values: {finite_fraction:.1%}")
if finite_values.size:
    print("Finite log10 reaction-time range:", np.log10(finite_values).min(), "to", np.log10(finite_values).max())

# 6.2 Priors on the log scale

What prior assumptions make sense when the linear predictor is on the log reaction-time scale?

A baseline reaction time around 250 ms corresponds to $\log(250)\approx5.52$. The chapter uses broad priors around this log scale.

In [ ]:
priors = {
    "Intercept": bmb.Prior("Normal", mu=5, sigma=0.55),
    "Days": bmb.Prior("Normal", mu=0, sigma=0.20),
    "sigma": bmb.Prior("Exponential", lam=3),
}
model = bmb.Model(
    "Reaction ~ Days", sleep, family="lognormal", priors=priors, center_predictors=False
)
model

In [ ]:
prior = model.prior_predictive(draws=500, random_seed=RANDOM_SEED)
azp.plot_ppc_dist(
    prior,
    group="prior_predictive",
    var_names=["Reaction"],
    kind="ecdf",
    figure_kwargs={"figsize": (7, 4)},
);

# 6.3 Population effect under a lognormal likelihood

What population-average deprivation effect is estimated under a positive, right-skewed likelihood?

In [ ]:
idata = model.fit(draws=1000, tune=1500, chains=4, target_accept=0.92, random_seed=RANDOM_SEED)
print("Divergences:", int(idata["sample_stats"]["diverging"].sum().item()))


In [ ]:
azs.summary(idata, var_names=["Intercept", "Days", "sigma"], ci_prob=0.90, ci_kind="hdi", round_to=2)

In [ ]:
azp.plot_trace_dist(idata, var_names=["Intercept", "Days", "sigma"]);

In [ ]:
bmb.interpret.plot_predictions(
    model,
    idata,
    conditional="Days",
    average_by="Subject",
    target="Reaction",
    prob=[0.50, 0.90],
)

# 6.4 Predictive consequences of the likelihood

Does changing the likelihood to lognormal address the predictive problems of the Gaussian regression?

In [ ]:
model.predict(
    idata,
    kind="response",
    inplace=True,
    random_seed=RANDOM_SEED,
)

azp.plot_ppc_dist(
    idata,
    var_names=["Reaction"],
    kind="ecdf",
    figure_kwargs={"figsize": (7, 4)},
);

# 6.5 Sensitivity on a transformed scale

How sensitive is the lognormal fit to the priors chosen on the transformed parameter scale?

In [ ]:
model.compute_log_likelihood(idata)
model.compute_log_prior(idata)

azs.psense_summary(idata, var_names=["Intercept", "Days", "sigma"])

azp.plot_psense_dist(
    idata,
    var_names=["Intercept", "Days", "sigma"],
    visuals={"dist": False},
);

A positive-only likelihood is scientifically attractive, but the main between-person structure is still missing. We next combine the lognormal response with varying participant intercepts and slopes.